# Missing values, duplicates and data types

**Video 6.3** | ShopKart project

**Goal:** Clean the raw ShopKart orders exactly like we did in Excel in Module 2 - but in a few lines of pandas - and save orders_clean.csv.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`, `products.csv` (used to look up the quantity-99 products).
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

## The cleaning plan
Every fix needs a **reason**. We never delete or fill data just to make the numbers look neat.

| Problem | Decision | Reason |
|---|---|---|
| 55 duplicate rows | Drop | Same order counted twice inflates revenue |
| City spellings | Strip, Title Case, rename old names | 'Bombay' and 'Mumbai' are the same market |
| unit_price "₹1,299" | Remove ₹ and comma, make number | Maths needs numbers |
| order_date two formats | Parse each exact format separately | Needed for months; the dayfirst shortcut silently flips ISO dates |
| payment_mode "upi" / blank | "upi"→"UPI", blank→"Unknown" | We do not know how they paid - do not guess |
| discount blank | Fill 0 | The business confirmed blank means no discount |
| returned Yes/No | Make Y/N | One code per answer |
| quantity 99 | Set to 1 | Data-entry typo; 10 Smartphones are real bulk orders, keep |
| rating blank | **Keep as missing** | "Not rated" is real information; filling would invent opinions |

In [ ]:
import pandas as pd
df = pd.read_csv("orders_raw.csv")
print(df.shape)

## Step 1 - Duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
# TODO: drop the duplicate rows
df = ...
print("After dropping:", df.shape)

## Step 2 - City names

In [ ]:
print("Before:", df["city"].nunique(), "spellings")
# TODO: strip spaces and make Title Case
df["city"] = df["city"].astype(str).str....
# TODO: rename the old city names
df["city"] = df["city"].replace({"New Delhi": "Delhi", ...})
print("After:", df["city"].nunique())
df["city"].value_counts()

## Step 3 - unit_price: text to number

In [ ]:
# TODO: remove "₹" and "," then convert to number
df["unit_price"] = (df["unit_price"].astype(str)
                    .str.replace("₹", "", regex=False)
                    .str.replace(..., ..., regex=False)
                    .astype(float).astype(int))
print(df["unit_price"].dtype)

## Step 4 - order_date: text to datetime
Two formats live in one column: most rows are `YYYY-MM-DD` (e.g. `2025-01-02`), 120 rows are `DD-MM-YYYY` (e.g. `02-01-2025`).

### The dayfirst trap
A tempting shortcut is `pd.to_datetime(df["order_date"], format="mixed", dayfirst=True)`. It runs without any error - but `dayfirst=True` is applied to **every** row, including the ISO ones. So `2025-01-02` (2 January) is read as **1 February**. No error, no warning, just wrong months.

In [ ]:
# See the trap on three example dates
demo = pd.Series(["2025-01-02", "2025-03-04", "02-01-2025"])
print(pd.to_datetime(demo, format="mixed", dayfirst=True))   # 1st and 2nd are WRONG (Feb 1, Apr 3)

In [ ]:
# How many of OUR rows would the shortcut silently flip?
trap = pd.to_datetime(df["order_date"], format="mixed", dayfirst=True)
iso = df["order_date"].str.match(r"^\d{4}-")
flipped = (trap[iso].dt.strftime("%Y-%m-%d") != df.loc[iso, "order_date"]).sum()
print("ISO dates read with day and month swapped:", flipped, "of", len(df))

Almost 2,000 orders would land in the wrong month - and every monthly chart and "best month" answer would be wrong.

### The safe way: one exact format at a time
1. Try the ISO format `%Y-%m-%d` on every row. Rows in the other format fail and become `NaT` (missing date) because of `errors="coerce"`.
2. Fill those gaps by trying `%d-%m-%Y`.

In [ ]:
print("DD-MM-YYYY rows:", df["order_date"].str.contains(r"^\d{2}-").sum())
s_ = df["order_date"].astype(str)
# TODO: try format "%Y-%m-%d" first, then fill the gaps with format "%d-%m-%Y"
df["order_date"] = pd.to_datetime(s_, format=..., errors="coerce").fillna(
                   pd.to_datetime(s_, format=..., errors="coerce"))
print(df["order_date"].dtype)
print("Dates that failed both formats:", df["order_date"].isnull().sum())   # should be 0

## Step 5 - payment_mode

In [ ]:
# TODO: "upi" -> "UPI", then fill blanks with "Unknown"
df["payment_mode"] = ...
df["payment_mode"].value_counts()

## Step 6 - discount: blank means 0

In [ ]:
print("Blank discounts:", df["discount"].isnull().sum())
# TODO: fill blank discount with 0.0
df["discount"] = ...
print("After:", df["discount"].isnull().sum())

## Step 7 - returned: one code

In [ ]:
# TODO: map Yes->Y and No->N
df["returned"] = ...
df["returned"].value_counts()

## Step 8 - quantity: typo or real outlier?

In [ ]:
products = pd.read_csv("products.csv")
big = df[df["quantity"] >= 10].merge(products[["product_id", "product_name"]], on="product_id")
big[["order_id", "product_name", "quantity", "unit_price"]]

The 99s are cheap everyday items (a typo for 1). The three 10s are all Smartphones at ₹14,999 - a genuine bulk order. We fix the 99s and **keep** the 10s.

In [ ]:
# TODO: set quantity to 1 where it is 99
df.loc[..., "quantity"] = 1
df["quantity"].value_counts().sort_index()

## Step 9 - rating: leave the gaps

In [ ]:
print("Missing ratings:", df["rating"].isnull().sum())
print("Average rating (pandas skips blanks):", round(df["rating"].mean(), 2))

If we filled blanks with 0, the average would fall and we would claim customers are unhappy when they simply did not rate. If we filled with the average, we would invent opinions. So we keep them as NaN.

## Final check and save

In [ ]:
print(df.shape)
print(df.isnull().sum())
df.info()

In [ ]:
# TODO: save to orders_clean.csv without the index
df.to_csv(...)

## Practice
Using the cleaned `df`: what share of orders now have payment_mode "Unknown"? How many orders came from Pune?

In [ ]:
# TODO: practice
...